# 02 — Feature Engineering

Builds candidate-edge features and training-ready tables in `data/processed/features/`. **Requires:** notebook 01 outputs and `src/data/feature_engineering.py`.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)

working_directory = Path.cwd().resolve()
PROJECT_ROOT = next(
    (c for c in [working_directory, *working_directory.parents]
     if (c / "data" / "raw" / "train").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate a project root containing data/raw/train.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CLEANED_DIR = PROJECT_ROOT / "data" / "processed" / "cleaned"
FEATURE_DIR = PROJECT_ROOT / "data" / "processed" / "features"
TIMESTAMP_TO_SECONDS = 1.0
print("Project root:", PROJECT_ROOT)


## Candidate-edge features and training-ready tables



In [ ]:
import importlib
import sys

source_directory = str(PROJECT_ROOT / "src")
if source_directory not in sys.path:
    sys.path.insert(0, source_directory)

import data.feature_engineering as feature_engineering
feature_engineering = importlib.reload(feature_engineering)
FEATURE_COLUMNS = feature_engineering.FEATURE_COLUMNS
prepare_model_data = feature_engineering.prepare_model_data

feature_audit = prepare_model_data(
    cleaned_root=CLEANED_DIR,
    output_root=FEATURE_DIR,
    validation_fraction=0.2,
    random_seed=42,
    max_candidates=32,
    nearest_samples=128,
    sample_spacing_m=30.0,
    label_radius_m=50.0,
    ambiguity_margin_m=5.0,
    timestamp_to_seconds=TIMESTAMP_TO_SECONDS,
)

training_trip_ids = set(
    feature_audit.loc[feature_audit["split"] == "train", "trip_id"]
)
validation_trip_ids = set(
    feature_audit.loc[feature_audit["split"] == "validation", "trip_id"]
)
assert training_trip_ids.isdisjoint(validation_trip_ids)
assert (
    set(feature_audit.loc[feature_audit["partition"] == "student_test", "split"])
    == {"student_test"}
)
assert set(FEATURE_COLUMNS).isdisjoint(
    {"candidate_arc_id", "label_distance_m", "label_margin_m", "route_geometry_suspect", "route_sequence_suspect", "route_sequence_consistent", "is_target"}
)

print("Feature columns:", FEATURE_COLUMNS)
print("Training trips:", len(training_trip_ids))
print("Validation trips:", len(validation_trip_ids))
print(
    "Minimum high-confidence candidate recall:",
    feature_audit.loc[
        feature_audit["partition"] == "train",
        "candidate_recall_high_confidence",
    ].min(),
)
display(
    feature_audit.groupby(["partition", "split"], as_index=False).agg(
        trips=("trip_id", "count"),
        gps_observations=("gps_observations", "sum"),
        candidate_rows=("candidate_rows", "sum"),
        supervised_observations=("supervised_observations", "sum"),
        geometry_suspect_trips=("route_geometry_suspect", "sum"),
        sequence_suspect_trips=("route_sequence_suspect", "sum"),
        sequence_labels_dropped=("route_sequence_labels_dropped", "sum"),
        median_label_distance_m=("median_label_distance_m", "median"),
        minimum_candidate_recall=("candidate_recall_high_confidence", "min"),
    )
)

quality_review = feature_audit.loc[
    feature_audit["route_geometry_suspect"]
    | feature_audit["route_sequence_suspect"]
    | (feature_audit["supervised_observations"] == 0),
    [
        "trip_id",
        "partition",
        "split",
        "route_geometry_suspect",
        "route_sequence_suspect",
        "high_confidence_labels",
        "ambiguous_high_confidence_labels",
        "candidate_recall_high_confidence",
        "route_sequence_labels_dropped",
        "supervised_observations",
    ],
]
display(quality_review)
